# NASDAQ TotalView-ITCH: Order Book Data Parsing

**Chapter 3: Market Microstructure**

**Docker image**: `ml4t`

## Purpose

This notebook demonstrates how to parse NASDAQ's TotalView-ITCH binary protocol.
Understanding MBO (message-by-order) data is foundational for microstructure-based ML features.

## Learning Objectives

After completing this notebook, you will be able to:
- Read one ITCH v5.0 message out of a binary file: find where it starts, how long it is,
  and which of the twenty-odd message types it is.
- Unpack a message's fields with Python's `struct` module, and convert the two encodings
  ITCH uses - a nanosecond offset from midnight, and a price held as an integer with four
  implied decimal places.
- Run the parse over a full trading day, writing each message type to its own Parquet
  partition so that a day that does not fit in memory still lands on disk.
- Load an already-parsed day and check that every message type reads back.
- Say when the Python parser is the right tool and when the Rust one is.

## Book reference

Section §3.3, *From raw messages to the limit order book* - the binary-parsing-at-scale
subsection and Table 3.1. §3.2 places ITCH among the other feeds.

## Cross-References

- **Downstream**: `02_itch_lob_reconstruction` (builds order book from these messages)
- **Related**: `09_databento_mbo_analysis` (alternative MBO data source)

## Data Requirements

ITCH sample data can be downloaded using:
```bash
python data/equities/market/microstructure/nasdaq_itch_download.py --list   # List available files
python data/equities/market/microstructure/nasdaq_itch_download.py          # Download default sample
```

Files are ~5GB compressed from: https://emi.nasdaq.com/ITCH/

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# NASDAQ TotalView-ITCH：委託簿資料解析

**第三章：市場微觀結構**

**Docker 映像檔**：`ml4t`

## 目的

本 Notebook 示範如何解析 NASDAQ 的 TotalView-ITCH 二進位協定。
理解 MBO（逐筆委託訊息）資料，是建立市場微觀結構機器學習特徵的基礎。

## 學習目標

完成本 Notebook 後，你將能夠：
- 從二進位檔案讀取一則 ITCH v5.0 訊息：找出其起始位置、長度，以及所屬的二十多種訊息類型。
- 使用 Python 的 `struct` 模組解開訊息欄位，並轉換 ITCH 使用的兩種編碼：自午夜起算的奈秒偏移量，以及隱含四位小數的整數價格。
- 解析完整交易日，將各種訊息類型寫入各自的 Parquet 分割區，讓無法容納於記憶體的一日資料仍能寫入磁碟。
- 載入已解析的一日資料，並檢查每種訊息類型都能讀回。
- 說明何時適合使用 Python 解析器，何時適合使用 Rust 解析器。

## 書籍對照

第 §3.3 節，〈從原始訊息到限價委託簿〉中的大規模二進位解析小節與表 3.1。第 §3.2 節說明 ITCH 在其他資料饋送中的定位。

## 交叉參照

- **下游**：`02_itch_lob_reconstruction`（從這些訊息建立委託簿）
- **相關**：`09_databento_mbo_analysis`（另一種 MBO 資料來源）

## 資料需求

可使用以下指令下載 ITCH 範例資料：
```bash
python data/equities/market/microstructure/nasdaq_itch_download.py --list   # List available files
python data/equities/market/microstructure/nasdaq_itch_download.py          # Download default sample
```

檔案壓縮後約為 5GB，來源：https://emi.nasdaq.com/ITCH/
<!-- ML4T_ZH_END c0000-p000 -->


## ITCH Message Types

The [ITCH v5.0 specification](https://www.nasdaqtrader.com/content/technicalsupport/specifications/dataproducts/NQTVITCHSpecification.pdf) defines 20+ message types:

| Type | Name | Description |
|------|------|-------------|
| **S** | System Event | Market open/close events |
| **R** | Stock Directory | Ticker information and characteristics |
| **H** | Trading Action | Trading halts, pauses, and resumptions |
| **Y** | Reg SHO Restriction | Short sale price test restrictions |
| **L** | Market Participant | Market maker positions |
| **V** | MWCB Decline Level | Market-wide circuit breaker levels |
| **W** | MWCB Status | Circuit breaker breach status |
| **A** | Add Order | New limit order enters the book |
| **F** | Add Order (MPID) | Same as A, with market participant ID |
| **E** | Order Executed | Partial/full execution against standing order |
| **C** | Order Executed w/Price | Execution at different price (hidden orders) |
| **X** | Order Cancel | Partial cancellation |
| **D** | Order Delete | Full removal from book |
| **U** | Order Replace | Modify price/size (cancel + add) |
| **P** | Trade | Non-displayed execution |
| **Q** | Cross Trade | Opening/closing cross |
| **B** | Broken Trade | Trade cancellation |
| **I** | NOII | Net Order Imbalance Indicator (auction) |
| **J** | LULD Auction Collar | Limit up-limit down price bands |
| **K** | IPO Quoting Period | IPO quotation timing |

By combining these messages chronologically, we can reconstruct the order book at any point in time.

<!-- ML4T_ZH_BEGIN c0001-p000 -->
## ITCH 訊息類型

[ITCH v5.0 規格](https://www.nasdaqtrader.com/content/technicalsupport/specifications/dataproducts/NQTVITCHSpecification.pdf) 定義了 20 多種訊息類型：

| 類型 | 名稱 | 說明 |
|------|------|-------------|
| **S** | 系統事件 | 市場開盤／收盤事件 |
| **R** | 股票目錄 | 股票代碼資訊與特性 |
| **H** | 交易狀態措施 | 交易停止、暫停與恢復 |
| **Y** | Reg SHO 限制 | 放空價格測試限制 |
| **L** | 市場參與者 | 造市商部位 |
| **V** | MWCB 跌幅門檻 | 全市場熔斷門檻 |
| **W** | MWCB 狀態 | 熔斷門檻觸發狀態 |
| **A** | 新增委託（Add Order） | 新的限價委託進入委託簿 |
| **F** | 新增委託（MPID） | 與 A 相同，另含市場參與者識別碼 |
| **E** | 委託成交執行（Order Executed） | 對簿上委託進行部分／全部成交執行 |
| **C** | 委託成交執行（附價格） | 以不同價格成交執行（隱藏委託） |
| **X** | 取消委託（Order Cancel） | 部分取消 |
| **D** | 刪除委託（Order Delete） | 從委託簿完全移除 |
| **U** | 替換委託（Order Replace） | 修改價格／數量（取消 + 新增） |
| **P** | 成交 | 未揭示的成交執行 |
| **Q** | 集合競價成交 | 開盤／收盤集合競價 |
| **B** | 作廢成交 | 取消成交 |
| **I** | NOII | 淨委託失衡指標（集合競價） |
| **J** | LULD 集合競價價格界限 | 漲跌幅限制價格帶 |
| **K** | IPO 報價期間 | IPO 報價時程 |

依時間順序整合這些訊息，即可重建任意時點的委託簿。
<!-- ML4T_ZH_END c0001-p000 -->


In [1]:
"""NASDAQ TotalView-ITCH: Order Book Data Parsing — parse ITCH binary protocol into structured messages."""

import gzip
import os
import shutil
import struct
from collections import Counter, defaultdict
from datetime import date, datetime
from pathlib import Path
from time import time

import polars as pl
from itch_message_specs import (
    FMT_DICT,
    MESSAGE_SPECS,
    NT_DICT,
    flush_to_parquet,
    parse_price4,
    parse_timestamp,
    print_message_formats,
)
from tqdm.auto import tqdm

from data import load_nasdaq_itch
from utils.paths import display_path

In [2]:
SKIP_PARSING = False
# Messages to read before stopping. None parses the whole session, which is what the
# shipped output is built from; a small number is how you try the parser out first.
MAX_MESSAGES = None

The parse reads one directory and writes another. `load_nasdaq_itch(get_base_path=True)`
resolves the parsed-message root from the repository's data configuration rather than a
path typed here, so the same notebook runs against a local checkout and inside the Docker
image. The raw binary sits in a `raw/` directory beside it, which is where the download
script puts it.

`must_exist=False` is what makes a clean start work. For every other notebook in this
chapter an absent `messages/` directory is the download instruction and the loader is
right to refuse; this notebook is the one that creates it, and it has to be told where
before there is anything there. The directory itself is created in Section 4, where the
parse is about to write: creating it here would leave an empty one behind on a machine
with no raw feed, and an empty directory is exactly what stops the loader refusing.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
解析作業從一個目錄讀取資料，並寫入另一個目錄。`load_nasdaq_itch(get_base_path=True)` 從儲存庫的資料設定解析出已解析訊息的根目錄，而非使用此處手動輸入的路徑，因此同一份 Notebook 可在本機檢出目錄與 Docker 映像檔內執行。原始二進位檔位於旁邊的 `raw/` 目錄，下載腳本會將檔案放在那裡。

`must_exist=False` 讓全新環境能順利啟動。對本章其餘 Notebook 而言，缺少 `messages/` 目錄就表示需要下載資料，載入器拒絕載入是正確的；本 Notebook 則負責建立該目錄，必須在其中尚無資料時先告知位置。目錄本身會在第 4 節解析作業即將寫入時建立：若在此處建立，沒有原始資料饋送的機器上就會留下空目錄，而空目錄恰好會使載入器不再拒絕載入。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
MESSAGE_DIR = load_nasdaq_itch(get_base_path=True, must_exist=False)
ITCH_RAW_DIR = MESSAGE_DIR.parent / "raw"

print(f"Raw ITCH binary (input):  {display_path(ITCH_RAW_DIR)}")
print(f"Parsed messages (output): {display_path(MESSAGE_DIR)}")
raw_present = ITCH_RAW_DIR.exists() and next(ITCH_RAW_DIR.iterdir(), None) is not None
print(f"Raw binary present: {raw_present}")
if not raw_present:
    print(
        "  Fetch it first: "
        "uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
    )


def message_type_dirs() -> list[Path]:
    """Parsed message-type subdirectories, one uppercase letter each.

    Empty before Section 4 has run: `messages/` itself does not exist on a machine that
    has never parsed, so every cell that lists message types goes through here.
    """
    if not MESSAGE_DIR.exists():
        return []
    return [
        d
        for d in sorted(MESSAGE_DIR.iterdir())
        if d.is_dir() and len(d.name) == 1 and d.name.isupper()
    ]

Raw ITCH binary (input):  <ML4T_DATA_PATH>/equities/market/microstructure/nasdaq_itch/raw
Parsed messages (output): <ML4T_DATA_PATH>/equities/market/microstructure/nasdaq_itch/messages
Raw binary present: True


## 1. Message Specifications

Each ITCH message has a fixed binary structure. The format is defined in `itch_message_specs.py`
using Python's `struct` module. Format codes:
- `H` = unsigned short (2 bytes)
- `I` = unsigned int (4 bytes)
- `Q` = unsigned long long (8 bytes)
- `s` = char (1 byte), `Ns` = N chars
- `>` = big-endian byte order

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 1. 訊息規格

每則 ITCH 訊息都有固定的二進位結構。格式在 `itch_message_specs.py` 中使用 Python 的 `struct` 模組定義。格式代碼：
- `H` = 無號短整數（2 位元組）
- `I` = 無號整數（4 位元組）
- `Q` = 無號長長整數（8 位元組）
- `s` = 字元（1 位元組），`Ns` = N 個字元
- `>` = 大端序位元組順序
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
# Show message formats (loaded from utils/itch_message_specs.py)
print_message_formats()

ITCH Message Formats:

  S (System Event             ): 11 bytes - >HH6ss
  R (Stock Directory          ): 38 bytes - >HH6s8sssIss2ssssssIs
  A (Add Order                ): 35 bytes - >HH6sQsI8sI
  F (Add Order MPID           ): 39 bytes - >HH6sQsI8sI4s
  E (Order Executed           ): 30 bytes - >HH6sQIQ
  C (Order Executed with Price): 35 bytes - >HH6sQIQsI
  X (Order Cancel             ): 22 bytes - >HH6sQI
  D (Order Delete             ): 18 bytes - >HH6sQ
  U (Order Replace            ): 34 bytes - >HH6sQQII
  P (Trade                    ): 43 bytes - >HH6sQsI8sIQ
  Q (Cross Trade              ): 39 bytes - >HH6sQ8sIQs
  H (Trading Action           ): 24 bytes - >HH6s8sss4s
  Y (Reg SHO Restriction      ): 19 bytes - >HH6s8ss
  L (Market Participant Position): 25 bytes - >HH6s4s8ssss
  V (MWCB Decline Level       ): 34 bytes - >HH6sQQQ
  W (MWCB Status              ): 11 bytes - >HH6ss
  J (LULD Auction Collar      ): 34 bytes - >HH6s8sIIII
  K (IPO Quoting Period       ): 27 byte

## 2. Binary Parsing Example

Let's demonstrate how binary parsing works by creating and parsing a sample Add Order message.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 二進位解析範例

我們透過建立並解析一則新增委託訊息，示範二進位解析的運作方式。
<!-- ML4T_ZH_END c0008-p000 -->


Packing a message by hand is the quickest way to see the layout. An Add Order carries the
format `>HH6sQsI8sI`: big-endian, two unsigned shorts, a six-byte timestamp, an eight-byte
order reference, a one-character side, the share count, an eight-character padded ticker
and the price. Unpacking it below returns exactly these fields.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
手動封裝一則訊息，是理解其配置最快的方法。新增委託使用 `>HH6sQsI8sI` 格式：大端序、兩個無號短整數、六位元組的時間戳記、八位元組的委託參考編號、單一字元的買賣方向、股數、補齊至八個字元的股票代碼，以及價格。下方解開訊息後，傳回的正是這些欄位。
<!-- ML4T_ZH_END c0009-p000 -->


In [5]:
sample_add_order = struct.pack(
    ">HH6sQsI8sI",
    1234,  # stock_locate
    5678,  # tracking_number
    b"\x00\x00\x00\x00\x00\x01",  # timestamp (1 nanosecond)
    9876543210,  # order_reference_number
    b"B",  # buy_sell_indicator
    100,  # shares
    b"AAPL    ",  # stock (padded to 8 chars)
    1500000,  # price: four implied decimals, so this is $150
)

print(f"Raw Add Order message ({len(sample_add_order)} bytes):")
print(f"  Hex: {sample_add_order.hex()}")

Raw Add Order message (35 bytes):
  Hex: 04d2162e000000000001000000024cb016ea42000000644141504c202020200016e360


In [6]:
# Parse the binary data using our struct format
parsed = struct.unpack(FMT_DICT["A"], sample_add_order)
add_order = NT_DICT["A"]._make(parsed)

print("Parsed Add Order Message:")
print("-" * 40)
for field, value in add_order._asdict().items():
    if isinstance(value, bytes):
        value = value.decode("ascii").strip()
    print(f"  {field:25}: {value}")

Parsed Add Order Message:
----------------------------------------
  stock_locate             : 1234
  tracking_number          : 5678
  timestamp                :      
  order_reference_number   : 9876543210
  buy_sell_indicator       : B
  shares                   : 100
  stock                    : AAPL
  price                    : 1500000


In [7]:
# Apply conversions using helper functions from utils.itch_message_specs
ts_ns = parse_timestamp(add_order.timestamp)
price = parse_price4(add_order.price)

print(f"Timestamp: {ts_ns:,} nanoseconds = {ts_ns / 1e9:.9f} seconds after midnight")
print(f"Price: ${price:.4f}")

Timestamp: 1 nanoseconds = 0.000000001 seconds after midnight
Price: $150.0000


## 3. What the Pipeline Holds Right Now

Before parsing anything, take stock: the raw binary is the input, the per-message-type
Parquet directories are the output, and either can be absent. On a first run the parsed
side is empty, and it stays empty until Section 4 writes it. The parsed messages are
loaded in Section 5, after there is something to load.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
## 3. 管線目前持有哪些資料

解析任何資料之前，先盤點現況：原始二進位檔是輸入，依訊息類型分開的 Parquet 目錄是輸出，兩者都可能不存在。首次執行時，已解析資料端是空的，直到第 4 節寫入資料後才有內容。已解析訊息會在第 5 節有資料可載入之後才載入。
<!-- ML4T_ZH_END c0013-p000 -->


In [8]:
# Check what data is available locally
print("ITCH Data Pipeline Status:")
print("-" * 50)

# Step 1: Raw binary from download
raw_files = []
if ITCH_RAW_DIR.exists():
    raw_files = list(ITCH_RAW_DIR.glob("*.gz")) + list(ITCH_RAW_DIR.glob("*.bin"))
print(f"Raw binary files: {len(raw_files)}")
for f in raw_files:
    print(f"  {f.name} ({f.stat().st_size / 1e9:.2f} GB)")

# Step 2: Parsed messages (single uppercase letter = message type)
parsed_types = message_type_dirs()
parsed_with_data = [d for d in parsed_types if list(d.glob("*.parquet"))]
print(f"Parsed message types: {len(parsed_with_data)}")
for msg_dir in parsed_with_data:
    name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
    n_files = len(list(msg_dir.glob("*.parquet")))
    print(f"  {msg_dir.name} ({name}): {n_files} files")

ITCH Data Pipeline Status:
--------------------------------------------------
Raw binary files: 2
  01302020.NASDAQ_ITCH50.gz (5.60 GB)
  01302020.NASDAQ_ITCH50.bin (12.95 GB)
Parsed message types: 0


## 4. Full Parser Implementation

The parser below is written to be read. It processes one message at a time in Python, which
is what makes each step visible and also what makes a full trading day take about twenty
minutes. Section 6 covers the Rust parser, which emits the same Parquet schema and is what
you would run over many days.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
## 4. 完整解析器實作

下方解析器以便於閱讀為目標撰寫。它在 Python 中一次處理一則訊息，讓每個步驟都清楚可見，也因此解析完整交易日約需二十分鐘。第 6 節介紹 Rust 解析器，它輸出相同的 Parquet 結構描述，適合用於處理多日資料。
<!-- ML4T_ZH_END c0015-p000 -->


### Parser Helpers

The parse splits into three functions. `read_frame` takes the next message off the file
and hands back its type and its raw bytes; `decode_message` turns those bytes into a
dictionary of Python values; and `parse_itch_file` runs the loop, buffering decoded
messages and writing them out in batches.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
### 解析器輔助函式

解析作業分為三個函式。`read_frame` 從檔案取出下一則訊息，傳回其類型與原始位元組；`decode_message` 將這些位元組轉成包含 Python 值的字典；`parse_itch_file` 則執行迴圈，緩衝已解碼的訊息並分批寫出。
<!-- ML4T_ZH_END c0016-p000 -->


In [9]:
def read_frame(f, pbar) -> tuple[str, bytes] | None:
    """Read one ITCH message frame: 2-byte length + 1-byte type + payload.

    Returns (msg_type, payload) on success, or None on EOF/truncation.
    """
    # 2-byte big-endian length prefix (message size including type byte)
    length_bytes = f.read(2)
    if len(length_bytes) < 2:
        return None
    pbar.update(2)

    msg_size = int.from_bytes(length_bytes, "big")

    # 1-byte message type
    msg_type_byte = f.read(1)
    if len(msg_type_byte) < 1:
        print(f"\nWarning: Truncated message at byte {f.tell()}, expected type byte")
        return None
    pbar.update(1)

    msg_type = msg_type_byte.decode("ascii")

    # Payload (msg_size includes type byte, so payload is msg_size - 1)
    payload = f.read(msg_size - 1)
    if len(payload) < msg_size - 1:
        print(f"\nWarning: Truncated payload for message type {msg_type}")
        return None
    pbar.update(msg_size - 1)

    return msg_type, payload

Decode binary payload into a Python dict, converting raw timestamp bytes
to nanosecond integers and byte strings to stripped ASCII.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
將二進位酬載解碼成 Python 字典，把原始時間戳記位元組轉為奈秒整數，並將位元組字串轉為去除前後空白的 ASCII 字串。
<!-- ML4T_ZH_END c0018-p000 -->


In [10]:
def decode_message(msg_type: str, payload: bytes) -> dict | None:
    """Unpack binary payload into a dict, converting timestamps and strings.

    Returns parsed message dict, or None on struct error.
    """
    try:
        parsed = struct.unpack(FMT_DICT[msg_type], payload)
        msg = NT_DICT[msg_type]._make(parsed)._asdict()
    except struct.error:
        return None

    # Convert timestamp: nanoseconds since midnight
    if "timestamp" in msg:
        msg["timestamp"] = int.from_bytes(msg["timestamp"], "big")

    # Decode string fields
    for field, value in msg.items():
        if isinstance(value, bytes):
            msg[field] = value.decode("ascii").strip()

    return msg

The main parser reads the binary file sequentially, buffering decoded messages
and flushing to Parquet periodically to bound memory usage.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
主解析器依序讀取二進位檔案，緩衝已解碼的訊息，並定期將緩衝內容寫入 Parquet，以限制記憶體用量。
<!-- ML4T_ZH_END c0020-p000 -->


In [11]:
def parse_itch_file(
    itch_file: Path,
    trading_day: date,
    output_dir: Path,
    max_buffered_messages: int = 10_000_000,
    max_messages: int | None = None,
) -> dict[str, int]:
    """Parse ITCH binary file and store messages as Parquet.

    Args:
        itch_file: Path to binary ITCH file (.bin, not .gz).
        trading_day: Trading date for timestamp construction.
        output_dir: Directory for Parquet output (one subdir per message type).
        max_buffered_messages: Flush threshold (total buffered messages).
        max_messages: Optional limit for testing.

    Returns:
        Dictionary with message type counts.
    """
    # Midnight timestamp for the trading day (ITCH timestamps are nanoseconds offset)
    base_ts = datetime(trading_day.year, trading_day.month, trading_day.day)
    file_counters: dict[str, int] = defaultdict(int)

    buffers = defaultdict(list)
    counts = Counter()
    file_size = itch_file.stat().st_size
    start_time = time()

    with (
        itch_file.open("rb") as f,
        tqdm(total=file_size, desc="Parsing ITCH", unit="B", unit_scale=True) as pbar,
    ):
        while True:
            if max_messages and sum(counts.values()) >= max_messages:
                print(f"\nLimit reached: {max_messages:,} messages")
                break

            frame = read_frame(f, pbar)
            if frame is None:
                break
            msg_type, payload = frame
            counts[msg_type] += 1

            if msg_type not in FMT_DICT:
                continue

            msg = decode_message(msg_type, payload)
            if msg is None:
                continue

            # Check for end of messages
            if msg_type == "S" and msg.get("event_code") == "C":
                print("\nEnd of Messages")
                flush_to_parquet(buffers, output_dir, base_ts, file_counters)
                break

            buffers[msg_type].append(msg)

            # Periodic flush
            if sum(len(v) for v in buffers.values()) >= max_buffered_messages:
                flush_to_parquet(buffers, output_dir, base_ts, file_counters)

    # Final flush
    if any(buffers.values()):
        flush_to_parquet(buffers, output_dir, base_ts, file_counters)

    elapsed = time() - start_time
    total = sum(counts.values())
    print(f"Parsed {total:,} messages in {elapsed:.1f}s ({total / elapsed:,.0f} msg/s)")

    return dict(counts)

In [12]:
# Locate ITCH data file (skip if SKIP_PARSING is set)
if SKIP_PARSING:
    print("SKIP_PARSING=True: skipping ITCH binary parsing (uses pre-parsed data)")
    itch_file = None
    counts = {}
else:
    # Clear any existing parsed data to avoid schema conflicts
    # (Different parser versions may produce different schemas)
    # Set ITCH_KEEP_EXISTING=1 to skip cleanup and use existing data
    clear_existing = os.environ.get("ITCH_KEEP_EXISTING", "0") != "1"
    if clear_existing and MESSAGE_DIR.exists() and list(MESSAGE_DIR.glob("*/part-*.parquet")):
        print(f"Clearing existing parsed data in {MESSAGE_DIR}")
        print("  (Set ITCH_KEEP_EXISTING=1 to keep existing data)")
        shutil.rmtree(MESSAGE_DIR)
        MESSAGE_DIR.mkdir(parents=True, exist_ok=True)

    # Find ITCH file (compressed or uncompressed)
    gz_files = list(ITCH_RAW_DIR.glob("*.gz")) if ITCH_RAW_DIR.exists() else []
    bin_files = list(ITCH_RAW_DIR.glob("*.bin")) if ITCH_RAW_DIR.exists() else []

    if not gz_files and not bin_files:
        raise FileNotFoundError(
            f"No raw ITCH binary found at {ITCH_RAW_DIR}.\n"
            "Download first:\n"
            "  uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
        )

In [13]:
# Decompress if needed and extract trading date
if not SKIP_PARSING and (gz_files or bin_files):
    # Prefer uncompressed, otherwise decompress
    if bin_files:
        itch_file = bin_files[0]
        print(f"Found uncompressed: {itch_file.name}")
    else:
        gz_file = gz_files[0]
        itch_file = gz_file.with_suffix(".bin")

        if not itch_file.exists():
            print(f"Decompressing {gz_file.name}...")
            with gzip.open(gz_file, "rb") as f_in, open(itch_file, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"Created: {itch_file.name} ({itch_file.stat().st_size / 1e9:.1f} GB)")
        else:
            print(f"Found: {itch_file.name}")

    # Extract trading date from filename (format: MMDDYYYY.NASDAQ_ITCH50.bin)
    date_str = itch_file.stem.split(".")[0]
    if len(date_str) == 8 and date_str.isdigit():
        trading_day = date(int(date_str[4:8]), int(date_str[:2]), int(date_str[2:4]))
    else:
        trading_day = date(2020, 1, 30)  # Fallback

    print(f"Trading day: {trading_day}")

    # Parse ITCH file (full-day parse: ~22 min on the reference machine)
    if itch_file and itch_file.exists():
        # Created here, not where the path is resolved: an empty messages/ directory
        # left behind by a run that found no raw binary would stop load_nasdaq_itch
        # raising for every later notebook in the chapter.
        MESSAGE_DIR.mkdir(parents=True, exist_ok=True)
        counts = parse_itch_file(
            itch_file=itch_file,
            trading_day=trading_day,
            output_dir=MESSAGE_DIR,
            max_messages=MAX_MESSAGES,
        )

        print("\nMessage counts:")
        for msg_type, count in sorted(counts.items(), key=lambda x: -x[1]):
            name = MESSAGE_SPECS.get(msg_type, {}).get("name", "Unknown")
            print(f"  {msg_type} ({name:25}): {count:>10,}")

Found uncompressed: 01302020.NASDAQ_ITCH50.bin
Trading day: 2020-01-30


Parsing ITCH:   0%|          | 0.00/13.0G [00:00<?, ?B/s]


End of Messages


Parsed 423,285,709 messages in 2158.7s (196,084 msg/s)

Message counts:
  A (Add Order                ): 184,735,355
  D (Order Delete             ): 180,285,101
  U (Order Replace            ): 36,777,372
  E (Order Executed           ):  8,415,610
  X (Order Cancel             ):  4,990,972
  I (NOII                     ):  4,025,192
  F (Add Order MPID           ):  1,875,350
  P (Trade                    ):  1,779,727
  L (Market Participant Position):    216,802
  C (Order Executed with Price):    139,474
  Q (Cross Trade              ):     17,835
  Y (Reg SHO Restriction      ):      9,068
  H (Trading Action           ):      8,921
  R (Stock Directory          ):      8,916
  S (System Event             ):          6
  J (LULD Auction Collar      ):          5
  K (IPO Quoting Period       ):          2
  V (MWCB Decline Level       ):          1


## 5. Message Type Analysis

The parse has written one Parquet directory per message type, so the store can now be
read back. The cell below asserts on trade messages (`P`) because that is the type it
goes on to load. A full session always contains them, so an empty `P` directory means
the parse did not finish rather than that this day happened to have no trades. That
reasoning holds only for a full parse: a run with `MAX_MESSAGES` set can stop before
the first trade prints, so there the absence is reported and the remaining cells work
with whichever types the trial did write.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
## 5. 訊息類型分析

解析作業已為每種訊息類型寫出一個 Parquet 目錄，現在可以讀回儲存的資料。下方儲存格對成交訊息（`P`）執行斷言檢查，因為接著要載入的就是此類型。完整交易時段必定包含成交訊息，因此 `P` 目錄為空表示解析未完成，而非當天恰好沒有成交。這個推論僅適用於完整解析：設定了 `MAX_MESSAGES` 的執行可能在第一筆成交出現前就停止，因此這種情況會回報缺少該類型，其餘儲存格則使用試跑實際寫出的訊息類型。
<!-- ML4T_ZH_END c0024-p000 -->


In [14]:
trade_dir = MESSAGE_DIR / "P"
trade_files = sorted(trade_dir.glob("*.parquet")) if trade_dir.exists() else []

if not trade_files and MAX_MESSAGES is not None:
    print(
        f"No trade messages under {display_path(trade_dir)}: the parse stopped after "
        f"{MAX_MESSAGES:,} messages and a `P` need not appear that early. "
        "Set MAX_MESSAGES to None to read the whole session."
    )
else:
    assert trade_files, (
        f"No parsed trade messages at {trade_dir}.\n"
        "Section 4 writes them from the raw binary. If SKIP_PARSING is True, it was skipped\n"
        "and the store has to have been written by an earlier run or by the Rust parser of\n"
        "Section 6. If the raw binary is missing, fetch it first:\n"
        "  uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
    )

    trades = pl.read_parquet(trade_dir / "*.parquet")
    print(f"Loaded {len(trades):,} trade messages")
    print(f"Columns: {trades.columns}")

Loaded 1,779,727 trade messages
Columns: ['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number', 'buy_sell_indicator', 'shares', 'stock', 'price', 'match_number']


In [15]:
# Message type distribution — use lazy scan to count without loading all data
written = [d for d in message_type_dirs() if list(d.glob("*.parquet"))]
print("Parsed Message Types:")
print("-" * 50)
if not written:
    print("  Nothing parsed yet - Section 4 writes these from the raw binary.")
for msg_dir in written:
    count = pl.scan_parquet(msg_dir / "*.parquet").select(pl.len()).collect().item()
    name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
    print(f"  {msg_dir.name} ({name:25}): {count:>12,} messages")

Parsed Message Types:
--------------------------------------------------
  A (Add Order                ):  184,735,355 messages
  C (Order Executed with Price):      139,474 messages
  D (Order Delete             ):  180,285,101 messages
  E (Order Executed           ):    8,415,610 messages
  F (Add Order MPID           ):    1,875,350 messages
  H (Trading Action           ):        8,921 messages


  I (NOII                     ):    4,025,192 messages
  J (LULD Auction Collar      ):            5 messages
  K (IPO Quoting Period       ):            2 messages
  L (Market Participant Position):      216,802 messages
  P (Trade                    ):    1,779,727 messages
  Q (Cross Trade              ):       17,835 messages
  R (Stock Directory          ):        8,916 messages
  S (System Event             ):            5 messages
  U (Order Replace            ):   36,777,372 messages
  V (MWCB Decline Level       ):            1 messages
  X (Order Cancel             ):    4,990,972 messages
  Y (Reg SHO Restriction      ):        9,068 messages


In [16]:
# Schema compatibility check — verify we can read each message type
written = [d for d in message_type_dirs() if list(d.glob("*.parquet"))]
print("Schema Compatibility Check:")
print("-" * 50)
if not written:
    print("  Nothing parsed yet - Section 4 writes these from the raw binary.")
for msg_dir in written:
    try:
        sample = pl.scan_parquet(msg_dir / "*.parquet").head(5).collect()
        name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
        print(f"  [OK] {msg_dir.name} ({name:25}): cols={list(sample.columns)[:4]}...")
    except Exception as e:
        print(f"  [FAIL] {msg_dir.name}: {e}")

Schema Compatibility Check:
--------------------------------------------------
  [OK] A (Add Order                ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] C (Order Executed with Price): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] D (Order Delete             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] E (Order Executed           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] F (Add Order MPID           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] H (Trading Action           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...
  [OK] I (NOII                     ): cols=['stock_locate', 'tracking_number', 'timestamp', 'paired_shares']...
  [OK] J (LULD Auction Collar      ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']

  [OK] L (Market Participant Position): cols=['stock_locate', 'tracking_number', 'timestamp', 'mpid']...
  [OK] P (Trade                    ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] Q (Cross Trade              ): cols=['stock_locate', 'tracking_number', 'timestamp', 'shares']...
  [OK] R (Stock Directory          ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...
  [OK] S (System Event             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'event_code']...


  [OK] U (Order Replace            ): cols=['stock_locate', 'tracking_number', 'timestamp', 'original_order_reference_number']...
  [OK] V (MWCB Decline Level       ): cols=['stock_locate', 'tracking_number', 'timestamp', 'level_1']...
  [OK] X (Order Cancel             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] Y (Reg SHO Restriction      ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...


## 6. Production Parsing with Rust

The Python parser above decodes one message per loop iteration, and a trading day holds a
few hundred million of them. The same protocol parsed in Rust reads the file through a
memory map and unpacks each message without copying it first, so it neither pays the
per-message interpreter overhead nor holds the decoded messages in memory.

**Repository**: [github.com/ml4t/itch-parser](https://github.com/ml4t/itch-parser)

The figures in the book's Table 3.1, for the same 13 GB file on one machine, are about
twenty-three minutes and roughly 8 GB of memory in Python against under five minutes and
under 500 MB in Rust. Wall-clock timings move with disk and CPU, so read them as an order
of magnitude rather than a ratio: the gap is large enough that it decides which parser you
reach for, and not stable enough to quote to a decimal place. The throughput this notebook
printed above is the Python side of the same comparison, measured on the machine that ran
it.

### Installation

```bash
# Clone the repository
git clone https://github.com/ml4t/itch-parser.git
cd itch-parser

# Build release binary
cargo build --release
```

### Usage

```bash
# Parse ITCH file (works with .gz or uncompressed)
./target/release/itch_parser <input_file> <output_dir> <MMDDYYYY>

# Example
./target/release/itch_parser data/01302020.NASDAQ_ITCH50.gz ./messages 01302020
```

Output is identical Parquet files partitioned by message type, compatible with
the Python code in this notebook and downstream analysis.

### When to Use Which


<!-- ML4T_ZH_BEGIN c0028-p000 -->
## 6. 使用 Rust 進行正式環境解析

上方 Python 解析器每次迴圈解碼一則訊息，而一個交易日包含數億則訊息。使用 Rust 解析相同協定時，會透過記憶體映射讀取檔案，直接解開每則訊息而不先複製，因此既不承擔每則訊息的直譯器額外負擔，也不會將已解碼訊息保留在記憶體中。

**儲存庫**：[github.com/ml4t/itch-parser](https://github.com/ml4t/itch-parser)

書中表 3.1 的數據是在同一台機器上，針對相同的 13 GB 檔案測得：Python 約需二十三分鐘、約 8 GB 記憶體；Rust 則不到五分鐘、低於 500 MB。實際經過時間會隨磁碟與 CPU 改變，因此應將這些數據視為數量級的比較，而非固定比率：差距大到足以決定該使用哪個解析器，但並不穩定到能引用至小數位。本 Notebook 上方印出的吞吐量，是相同比較中的 Python 端數據，測量於實際執行它的機器。

### 安裝

```bash
# Clone the repository
git clone https://github.com/ml4t/itch-parser.git
cd itch-parser

# Build release binary
cargo build --release
```

### 使用方式

```bash
# Parse ITCH file (works with .gz or uncompressed)
./target/release/itch_parser <input_file> <output_dir> <MMDDYYYY>

# Example
./target/release/itch_parser data/01302020.NASDAQ_ITCH50.gz ./messages 01302020
```

輸出為相同的 Parquet 檔案，依訊息類型分割，可相容於本 Notebook 的 Python 程式碼及下游分析。

### 何時使用哪一種
<!-- ML4T_ZH_END c0028-p000 -->

| Use Case | Recommendation |
|----------|---------------|
| Learning the protocol | Python (this notebook) |
| Debugging parse issues | Python |
| Processing a single day | Either |
| Multi-day backtesting | **Rust** |
| Production pipeline | **Rust** |

Both parsers write the same Parquet schema, so a day parsed either way feeds every
notebook that follows without change.

<!-- ML4T_ZH_BEGIN c0028-p001 -->
| 使用情境 | 建議 |
|----------|---------------|
| 學習協定 | Python（本 Notebook） |
| 偵錯解析問題 | Python |
| 處理單日資料 | 兩者皆可 |
| 多日回測 | **Rust** |
| 正式環境管線 | **Rust** |

兩種解析器寫出相同的 Parquet 結構描述，因此無論以何種方式解析一日資料，都能直接供後續所有 Notebook 使用。
<!-- ML4T_ZH_END c0028-p001 -->


## Key Takeaways

1. **The protocol is message-by-order.** Every message that changes the book names the
   individual order it acts on, stamped to the nanosecond, which is what makes book
   reconstruction possible at all. Session-level messages - `S` for market events, `R`
   for the stock directory, `Q` for the auction crosses - carry no order reference and
   describe the venue rather than a single order.
2. **Seven message types carry the book.** `A` and `F` add an order, `E` and `C` execute
   against one, `X` cancels part of one, `D` deletes one, `U` replaces one.
3. **Numbers arrive encoded.** Prices are integers with four implied decimal places, and
   timestamps are nanoseconds since midnight, so both need converting before use.
4. **Parse in batches, not in one pass.** Buffering by message type and flushing to
   Parquet is what keeps a day that does not fit in memory from having to.
5. **Which parser depends on how many days you need.** Python reads clearly and takes
   about twenty minutes per day; Rust emits the same schema in a fraction of that.

### Known limitations

- The parse covers one venue. NASDAQ ITCH sees NASDAQ-routed activity, not the
  consolidated tape, so counts here are a venue's share of a symbol's trading rather than
  all of it.
- Message types outside `FMT_DICT` are counted and skipped rather than decoded.
- The Rust timings quoted above were measured elsewhere, on one machine; this notebook
  times only its own Python parse.

### Next Steps

- **Order Book Reconstruction**: `02_itch_lob_reconstruction`
- **Trading Activity Overview**: `05_itch_trading_activity` (includes E/C enrichment)

---

## Reference


<!-- ML4T_ZH_BEGIN c0029-p000 -->
## 重點整理

1. **此協定採逐筆委託訊息形式。** 每則改變委託簿的訊息都指明其作用的個別委託，並附有奈秒精度的時間戳記，這正是能夠重建委託簿的根本。交易時段層級的訊息——市場事件的 `S`、股票目錄的 `R`、集合競價成交的 `Q`——不含委託參考編號，描述的是交易場所而非單一委託。
2. **七種訊息類型支撐委託簿。** `A` 與 `F` 新增委託，`E` 與 `C` 對委託進行成交執行，`X` 取消部分委託，`D` 刪除委託，`U` 替換委託。
3. **數值以編碼形式傳入。** 價格是隱含四位小數的整數，時間戳記則是自午夜起算的奈秒數，因此使用前都需要轉換。
4. **分批解析，而非一次處理完畢。** 依訊息類型緩衝並寫入 Parquet，讓無法容納於記憶體的一日資料不必全部留在記憶體中。
5. **選擇解析器取決於所需處理的天數。** Python 易於閱讀，每日資料約需二十分鐘；Rust 只需其中一小部分時間，就能輸出相同的結構描述。

### 已知限制

- 解析範圍僅涵蓋一個交易場所。NASDAQ ITCH 觀察的是經 NASDAQ 路由的活動，而非整合成交行情，因此此處的計數是該場所占某股票交易的一部分，並非全部交易。
- `FMT_DICT` 以外的訊息類型只會計數並略過，不會解碼。
- 上述 Rust 耗時是在其他地方的一台機器上測得；本 Notebook 只對自身的 Python 解析計時。

### 後續步驟

- **委託簿重建**：`02_itch_lob_reconstruction`
- **交易活動概覽**：`05_itch_trading_activity`（包含 E/C 資訊補充）

---

## 參考文獻
<!-- ML4T_ZH_END c0029-p000 -->

Bouchaud, J.-P., Bonart, J., Donier, J., & Gould, M. (2018).
*Trades, Quotes and Prices: Financial Markets Under the Microscope*.
Cambridge University Press.
[https://doi.org/10.1017/9781009028943](https://doi.org/10.1017/9781009028943)

<!-- ML4T_ZH_BEGIN c0029-p001 -->
Bouchaud, J.-P., Bonart, J., Donier, J., & Gould, M. (2018).
*交易、報價與價格：顯微鏡下的金融市場*。
劍橋大學出版社。
[https://doi.org/10.1017/9781009028943](https://doi.org/10.1017/9781009028943)
<!-- ML4T_ZH_END c0029-p001 -->
